# Gemma: Credit-G — fine-tuning with missing values

Historical experiment source: `bench_gemma/FewZeroShot/LLM_Gemma3_4B_Fine_tune_Credit_G_with_missing_(ep=20).ipynb`. Outputs and stale result commentary were removed for publication. Scientific logic is retained; these experiments and their reported results have not been rerun or validated here.

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import train_test_split
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import time
import json
import random
from tqdm.auto import tqdm
import math
import gc

In [ ]:
import os
from huggingface_hub import login

if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

In [ ]:
MISSING_RATES = [0.0, 0.2, 0.5, 0.9]

prompt_config = {
    "task": "Classify credit risk as good or bad",
    "pos_label": "good",
    "neg_label": "bad",
    "entity": "Client",
    "question": "Is this client a good credit risk?"
}

openml_id = 31

base_output_dir = "/tmp/gemma_credit_g_missing"

# 1. Загрузка данных

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import LabelEncoder

def load_dataset(openml_id=1464):
    dataset = fetch_openml(data_id=openml_id, as_frame=True, parser='auto')
    X = dataset.data
    y = dataset.target

    df = X.copy()
    feature_names = X.columns.to_list()

    target_name = y.name
    df[target_name] = y

    if y.dtype == 'object' or y.dtype.name == 'category':
        le = LabelEncoder()
        df[target_name] = le.fit_transform(df[target_name])

    return df, feature_names, target_name


def split_dataset(df, target_name, test_size=0.2, val_size=0.25, seed=42):
    train_val_df, test_df = train_test_split(
        df, test_size=test_size, random_state=seed, stratify=df[target_name]
    )
    train_df, val_df = train_test_split(
        train_val_df, test_size=val_size, random_state=seed, stratify=train_val_df[target_name]
    )
    return train_df, val_df, test_df

df, feature_names, target_name = load_dataset(openml_id)
train_df, val_df, test_df = split_dataset(df, target_name)

df.head(5)

In [ ]:
df.info()

In [ ]:
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    counts = df[target_name].value_counts()
    pcts   = df[target_name].value_counts(normalize=True) * 100
    print(f"\n{name} (всего: {len(df)}):")
    print(f"  {prompt_config['neg_label']}: {counts[0]:5d} — {pcts[0]:.1f}%")
    print(f"  {prompt_config['pos_label']}: {counts[1]:5d} — {pcts[1]:.1f}%")

# 2. Вспомогательные функции

In [ ]:
def row_to_text_template_with_missing(row, feature_names, target_name, missing_rate=0.0, seed=None):

    rng = np.random.default_rng(seed)

    n_drop = int(len(feature_names) * missing_rate)
    dropped = set(rng.choice(feature_names, size=n_drop, replace=False)) if n_drop > 0 else set()

    template_parts = []

    for feature in feature_names:
        if feature in dropped:
            continue

        value = row[feature]

        if isinstance(value, (int, np.integer)):
            phrase = f"The value of {feature} is {value}."
        elif isinstance(value, (float, np.floating)):
            phrase = f"The value of {feature} is {value:.2f}."
        else:
            phrase = f"The category of {feature} is {value}."

        template_parts.append(phrase)

    text = " ".join(template_parts)

    return text

# Тест
test_row = train_df.iloc[0]
display(train_df.head(1))
for rate in MISSING_RATES:
    text_output = row_to_text_template_with_missing(
        row=test_row,
        feature_names=feature_names,
        target_name=target_name,
        missing_rate=rate
    )

    print(f"Missing Rate: {rate * 100:.0f}%")
    print(text_output)

In [ ]:
def parse_prediction(response, prompt_config):
    response = response.lower().strip().rstrip('.,!? ')
    pos, neg = prompt_config['pos_label'].lower(), prompt_config['neg_label'].lower()

    if response == pos: return 1
    if response == neg: return 0

    if response.startswith(pos): return 1
    if response.startswith(neg): return 0

    words = response.split()
    if pos in words: return 1
    if neg in words: return 0

    print(f"Warning: Could not parse '{response}' (expected '{prompt_config['pos_label']}' or '{prompt_config['neg_label']}')")
    return 0



response = prompt_config['pos_label']
pred = parse_prediction(response, prompt_config)
print(f"Response: '{response}'\nPrediction: {pred}\n")

response = "hi"
pred = parse_prediction(response, prompt_config)
print(f"Response: '{response}'\nPrediction: {pred}")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def compute_metrics(y_true, y_pred, y_prob=None):
    return {
        "ROC-AUC": roc_auc_score(y_true, y_prob if y_prob is not None else y_pred),
        "F1":       f1_score(y_true, y_pred, zero_division=0),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall":   recall_score(y_true, y_pred, zero_division=0),
    }

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.utils import resample

def bootstrap_metrics(y_true, y_pred, y_prob=None, n_iter=1000):
    scores = []

    for i in range(n_iter):
        if y_prob is not None:
            y_true_boot, y_pred_boot, y_prob_boot = resample(
                y_true, y_pred, y_prob, random_state=i+1
            )
        else:
            y_true_boot, y_pred_boot = resample(
                y_true, y_pred, random_state=i+1
            )
            y_prob_boot = None

        try:
            if y_prob_boot is not None:
                auc = roc_auc_score(y_true_boot, y_prob_boot)
            else:
                auc = 0.0

            f1 = f1_score(y_true_boot, y_pred_boot, zero_division=0)
            pr = precision_score(y_true_boot, y_pred_boot, zero_division=0)
            rc = recall_score(y_true_boot, y_pred_boot, zero_division=0)

            acc = accuracy_score(y_true_boot, y_pred_boot)
            scores.append((auc, f1, acc, pr, rc))

        except ValueError:
            continue

    scores = np.asarray(scores)
    means, stds = scores.mean(0), scores.std(0, ddof=1)
    names = ["ROC-AUC", "F1", "Accuracy", "Precision", "Recall"]

    return {n: f"{m:.4f}\u00b1{s:.4f}" for n, m, s in zip(names, means, stds)}

sample_row = test_df.iloc[0]
sample_text = row_to_text_template_with_missing(sample_row, feature_names, target_name)
print(f"\nПример текста:\n{sample_text[:300]}")

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

model_name = "google/gemma-3-4b-it"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map=device,
    attn_implementation="sdpa"
)

if torch.cuda.is_available():
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    used_mem = torch.cuda.memory_allocated() / 1e9
    print(f"GPU total memory: {total_mem:.2f} GB")
    print(f"GPU allocated memory: {used_mem:.2f} GB")

In [ ]:
def create_prompt_missing(row, feature_names, target_name, prompt_config, tokenizer, missing_rate=0.0, seed=None):

    system_prompt = (
        f"You are a classifier. {prompt_config['task']}: "
        f"'{prompt_config['pos_label']}' or '{prompt_config['neg_label']}'. "
        f"Answer with only one word: '{prompt_config['pos_label']}' or '{prompt_config['neg_label']}'."
    )

    input_text = row_to_text_template_with_missing(row, feature_names, target_name, missing_rate, seed)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",
         "content": f"{prompt_config['entity']} information: {input_text}\n{prompt_config['question']}"},
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

In [ ]:
import torch.nn.functional as F

BATCH_SIZE_EVAL = 32

def batched_rows(df, batch_size):
    for start in range(0, len(df), batch_size):
        yield df.iloc[start:start + batch_size]

def flush_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def predict_batch_with_prob(prompts, prompt_config, model, tokenizer, device, max_new_tokens=3):
    model_inputs = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True,
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=model_inputs.input_ids,
            attention_mask=model_inputs.attention_mask,
            max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            output_scores=True, return_dict_in_generate=True
        )

    input_len = model_inputs.input_ids.shape[1]
    del model_inputs

    generated_ids = outputs.sequences[:, input_len:]
    responses = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
    responses = [resp.strip().lower() for resp in responses]
    del generated_ids

    first_token_logits = outputs.scores[0]
    del outputs

    pos_id = tokenizer.encode(prompt_config['pos_label'], add_special_tokens=False)[0]
    neg_id = tokenizer.encode(prompt_config['neg_label'], add_special_tokens=False)[0]
    selected_logits = torch.stack([first_token_logits[:, pos_id], first_token_logits[:, neg_id]], dim=1)
    probs = F.softmax(selected_logits, dim=1)
    prob_pos = probs[:, 0].detach().cpu().tolist()
    del first_token_logits, selected_logits, probs

    flush_gpu()
    return responses, prob_pos

# Тест на маленьком батче
test_prompts = [
    create_prompt_missing(test_df.iloc[i], feature_names, target_name, prompt_config, tokenizer)
    for i in range(min(2, len(test_df)))
]
responses, probs = predict_batch_with_prob(test_prompts, prompt_config, base_model, tokenizer, device)
for i, (response, prob) in enumerate(zip(responses, probs)):
    print(f"Sample {i}: response='{response}', probability={prob:.4f}")

# 3. Fine-tuning с LoRA

## Подготовка данных для Fine-tuning

In [ ]:
def balance_dataset(train_df, target_name, prompt_config, seed=42):
    df_0 = train_df[train_df[target_name] == 0]
    df_1 = train_df[train_df[target_name] == 1]

    df_majority = df_0 if len(df_0) > len(df_1) else df_1
    df_minority = df_1 if len(df_0) > len(df_1) else df_0

    print(f"До балансировки:")
    print(f"{prompt_config['neg_label']}:  {len(df_majority)}")
    print(f"{prompt_config['pos_label']}: {len(df_minority)}")

    df_minority_up = resample(df_minority, replace=True,
                              n_samples=len(df_majority), random_state=seed)

    train_df_balanced = pd.concat([df_majority, df_minority_up])
    train_df_balanced = train_df_balanced.sample(frac=1, random_state=seed).reset_index(drop=True)

    print(f"\nПосле балансировки:")
    print(train_df_balanced[target_name].value_counts())

    return train_df_balanced

train_df_balanced = balance_dataset(train_df, target_name, prompt_config)

In [ ]:
def create_training_dataset(df, feature_names, target_name, prompt_config, tokenizer, missing_rate=0.0):
    system_prompt = (
        f"You are a classifier. {prompt_config['task']}: "
        f"'{prompt_config['pos_label']}' or '{prompt_config['neg_label']}'. "
        f"Answer with only one word: '{prompt_config['pos_label']}' or '{prompt_config['neg_label']}'."
    )
    texts = []

    for idx, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc=f"Dataset (missing={missing_rate:.0%})")):
        input_text = row_to_text_template_with_missing(row, feature_names, target_name, missing_rate, idx)
        target = prompt_config['pos_label'] if row[target_name] == 1 else prompt_config['neg_label']

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"{prompt_config['entity']} information: {input_text}\n{prompt_config['question']}"},
            {"role": "assistant", "content": target}
        ]

        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)

    return texts

In [ ]:
def finetune_model(missing_rate, train_df_balanced, feature_names, target_name,
                   prompt_config, tokenizer, base_model, device,
                   num_epochs=3, batch_size=16):

    output_dir = f"{base_output_dir}_missing{int(missing_rate * 100):03d}"
    print(f"  Fine-tuning: missing_rate = {missing_rate:.0%}")

    train_texts = create_training_dataset(
        train_df_balanced, feature_names, target_name,
        prompt_config, tokenizer, missing_rate=missing_rate)
    train_dataset = Dataset.from_dict({"text": train_texts})

    def tokenize_fn(examples):
        return tokenizer(examples["text"], truncation=True, max_length=512,
                         padding=False, return_token_type_ids=True)

    tokenized_dataset = train_dataset.map(tokenize_fn, batched=True,
                                          remove_columns=train_dataset.column_names)

    if "token_type_ids" not in tokenized_dataset.column_names:
        tokenized_dataset = tokenized_dataset.map(
            lambda batch: {"token_type_ids": [[0] * len(ids) for ids in batch["input_ids"]]},
            batched=True,
        )

    print(f"  Подготовлено {len(tokenized_dataset)} примеров для обучения")

    model_lora = AutoModelForCausalLM.from_pretrained(
        model_name, device_map=device,
        torch_dtype=torch.bfloat16, attn_implementation="sdpa"
    )
    model_lora.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )

    lora_config = LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
    )
    model_lora = get_peft_model(model_lora, lora_config)
    model_lora.print_trainable_parameters()

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=2,
        learning_rate=2e-4, bf16=True, tf32=True,
        logging_steps=10, save_strategy="no",
        optim="adamw_torch_fused", warmup_steps=50,
        max_grad_norm=1.0, weight_decay=0.01,
        report_to="none", dataloader_num_workers=4,
        dataloader_pin_memory=True, group_by_length=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        torch_compile=False,
    )

    trainer = Trainer(
        model=model_lora, args=training_args,
        train_dataset=tokenized_dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    )

    t0 = time.time()
    trainer.train()
    print(f"  Обучение завершено за {(time.time()-t0)/60:.1f} мин")

    del trainer
    flush_gpu()

    model_lora.eval()
    return model_lora

In [ ]:
def evaluate_on_test(model_finetuned, test_df, feature_names, target_name,
                     prompt_config, tokenizer, device,
                     eval_missing_rate=0.0):

    y_true, y_pred, y_prob = [], [], []
    t0 = time.time()
    n_batches = math.ceil(len(test_df) / BATCH_SIZE_EVAL)

    for batch_df in tqdm(batched_rows(test_df, BATCH_SIZE_EVAL), total=n_batches, desc="Test eval"):
        prompts = [
            create_prompt_missing(row, feature_names, target_name, prompt_config, tokenizer,
                                 missing_rate=eval_missing_rate)
            for _, row in batch_df.iterrows()
        ]
        responses, probs = predict_batch_with_prob(prompts, prompt_config, model_finetuned, tokenizer, device)

        for (_, row), response, prob in zip(batch_df.iterrows(), responses, probs):
            y_true.append(row[target_name])
            y_pred.append(parse_prediction(response, prompt_config))
            y_prob.append(prob)

        del prompts, responses, probs

    elapsed = time.time() - t0
    metrics = compute_metrics(np.array(y_true), np.array(y_pred), np.array(y_prob))
    boot = bootstrap_metrics(np.array(y_true), np.array(y_pred), np.array(y_prob), n_iter=1000)

    print("Метрики:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")
    print("Bootstrap (mean±std):")
    for k, v in boot.items():
        print(f"  {k}: {v}")

    return {
        "metrics": metrics,
        "bootstrap": boot,
        "time_total": elapsed,
        "time_per_sample": elapsed / len(y_true),
        "eval_missing_rate": eval_missing_rate,
    }

In [ ]:
all_results = {}

start_time = time.time()

for missing_rate in MISSING_RATES:
    tag = f"missing_{int(missing_rate * 100):03d}pct"
    print(f"\n{'='*60}")
    print(f"Эксперимент: missing_rate = {missing_rate:.0%}")
    print(f"{'='*60}")

    model_finetuned = finetune_model(
        missing_rate=missing_rate,
        train_df_balanced=train_df_balanced,
        feature_names=feature_names,
        target_name=target_name,
        prompt_config=prompt_config,
        tokenizer=tokenizer,
        base_model=base_model,
        device=device,
        num_epochs=20,
        batch_size=32,
    )

    result = evaluate_on_test(
        model_finetuned=model_finetuned,
        test_df=test_df,
        feature_names=feature_names,
        target_name=target_name,
        prompt_config=prompt_config,
        tokenizer=tokenizer,
        device=device,
        eval_missing_rate=missing_rate,
    )

    all_results[tag] = result

    del model_finetuned
    flush_gpu()

print(f"\nВсе эксперименты завершены за {(time.time()-start_time)/60:.1f} мин")

In [ ]:
all_results

In [ ]:
print("\nBootstrap (mean±std):")
for tag, res in all_results.items():
    rate = tag.replace("missing_", "").replace("pct", "%").lstrip("0") or "0%"
    print(f"\n  missing={rate}")
    for k, v in res["bootstrap"].items():
        print(f"    {k}: {v}")